# SI4006 · Proyecto final — Despliegue en Colab con enlace público

Hugging Face ya no permite Spaces de Gradio en CPU para cuentas gratuitas. Siguiendo la alternativa que dio el profesor,
la app (`final/space/app.py`) corre aquí y se publica con un **túnel de Cloudflare**. Así queda una URL pública que se
puede abrir desde cualquier dispositivo, con:

- `/` → la interfaz de Gradio: consulta de la documentación y **dictamen desde la imagen de una ficha** (OCR + clasificador de M1)
- `/health` → el health check (200 si el índice, el generador y la calculadora responden; 503 si no)
- un log JSON por petición (`final/space/logs/peticiones.jsonl` y la salida de la app)
- `/metrics` y la pestaña **Observabilidad** → latencia por petición, p50/p95, tasa de válvula y uso de la calculadora,
  calculados del log
- W&B: en línea si existe el secreto `WANDB_API_KEY` en Colab (llave 🔑 de la barra izquierda); si no, en modo
  **offline** (archivos locales en `final/space/wandb/`, sin cuenta), que se pueden subir después con `wandb sync`

Es la misma versión que medimos en el notebook de eficiencia: Qwen2.5-1.5B **int4 (Q4_K_M)** con llama.cpp en **CPU**.

**Cómo correrlo:** entorno **CPU** (no hace falta GPU) → *Ejecutar todo*. La primera vez tarda unos 5 a 10 minutos en
instalar y descargar el modelo. **La URL cambia cada vez que se corre** y deja de funcionar si se cierra la pestaña o
Colab desconecta la sesión. Para la sustentación hay que correrlo unos 15 minutos antes y dejar la pestaña abierta.

In [ ]:
import os, sys, time, json, subprocess, re
from pathlib import Path

REPO_URL = 'https://github.com/CamsMars/CREDIT'
# Si se subió CREDIT_repo.zip, se usa el zip (y se descarta un clon incompleto sin final/).
if Path('CREDIT_repo.zip').exists() and not Path('CREDIT/final/space/app.py').exists():
    subprocess.run('rm -rf CREDIT && unzip -q -o CREDIT_repo.zip', shell=True, check=True)
if not Path('CREDIT').exists():
    r = subprocess.run(['git', 'clone', '-q', REPO_URL], capture_output=True, text=True)
    if r.returncode != 0 or not Path('CREDIT/final/space/app.py').exists():
        print('No se pudo clonar con final/. Sube CREDIT_repo.zip al panel de archivos y corre: !unzip -q CREDIT_repo.zip')
APP = Path('CREDIT/final/space').resolve()
assert (APP / 'app.py').exists(), 'Falta la carpeta CREDIT/final/space'
print('App en', APP)

In [ ]:
# Colab ya trae torch, transformers, sentence-transformers, gradio y fastapi. Solo faltan estos.
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa > /dev/null
%pip install -q rank_bm25 wandb uvicorn pytesseract peft
# Colab trae torchao 0.10, que rompe peft al cargar el adaptador LoRA de M1; el proyecto no lo usa.
!pip uninstall -y -q torchao
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python==0.3.36', '--prefer-binary',
                    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cpu'],
                   capture_output=True, text=True)
print(r.stderr[-800:] or 'llama-cpp-python instalado')

In [ ]:
# W&B (opcional): lee el secreto de Colab. Sin él, la app funciona igual y solo escribe el log local.
entorno = dict(os.environ, CREDIT_VERSION='q4_k_m', CREDIT_THREADS=str(os.cpu_count()), PYTHONUNBUFFERED='1')
try:
    from google.colab import userdata
    entorno['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B: secreto encontrado')
except Exception as e:
    print('Sin secreto WANDB_API_KEY: W&B queda en modo offline (local, sin cuenta).')

In [ ]:
# Arranca la app en segundo plano y espera a que el health check diga ok.
import urllib.request
salida_app = open('/content/app_stdout.log', 'w')
app_proc = subprocess.Popen([sys.executable, 'app.py'], cwd=APP, env=entorno, stdout=salida_app, stderr=subprocess.STDOUT)

def health(url='http://localhost:7860/health'):
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read() or b'{}')
    except Exception as e:
        return None, {'status': type(e).__name__}

t0 = time.time()
while time.time() - t0 < 1200:
    codigo, cuerpo = health()
    print(f'{time.time()-t0:5.0f} s · {codigo} · {cuerpo.get("status")}', end='\r')
    if codigo == 200 or cuerpo.get('status') == 'error' or app_proc.poll() is not None:
        break
    time.sleep(10)
print()
print(json.dumps(cuerpo, indent=1, ensure_ascii=False))
if codigo != 200:
    print(open('/content/app_stdout.log').read()[-3000:])

In [ ]:
# Túnel de Cloudflare: publica localhost:7860 en una URL https://<algo>.trycloudflare.com
if not Path('/content/cloudflared').exists():
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('/content/cloudflared', 0o755)
tunel_log = open('/content/tunel.log', 'w')
tunel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:7860'],
                         stdout=tunel_log, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunel.log').read())
    if m:
        URL = m.group(0); break
print('URL pública :', URL)
print('Health check:', f'{URL}/health')

In [ ]:
# Comprobación desde afuera (a través del túnel). Puede tardar unos segundos en propagarse.
time.sleep(5)
for i in range(6):
    codigo, cuerpo = health(f'{URL}/health')
    if codigo == 200: break
    time.sleep(5)
print(codigo, json.dumps(cuerpo, ensure_ascii=False))
print(f'\nAbre {URL} en el celular para comprobar que se ve desde otro dispositivo.')

In [ ]:
# Los últimos registros JSON (uno por petición). Ejecutar de nuevo después de hacer preguntas en la app.
ruta = APP / 'logs' / 'peticiones.jsonl'
print(''.join(open(ruta).readlines()[-5:]) if ruta.exists() else 'Todavía no hay peticiones.')

In [ ]:
# Métricas de observabilidad (las mismas de la pestaña "Observabilidad" y de la ruta /metrics).
with urllib.request.urlopen(f'{URL}/metrics', timeout=20) as r:
    print(json.dumps(json.loads(r.read()), indent=1, ensure_ascii=False))
print(f'\nEn el navegador: {URL}/metrics')

In [ ]:
# Mantener la sesión viva mientras se usa la app (detener esta celda para terminar).
try:
    while True:
        codigo, cuerpo = health()
        print(time.strftime('%H:%M:%S'), codigo, cuerpo.get('status'), '· peticiones:', cuerpo.get('peticiones'), end='\r')
        time.sleep(60)
except KeyboardInterrupt:
    print('\nDetenido. La app y el túnel siguen corriendo hasta que se reinicie el entorno.')